In [1]:
dict_sample = {
    "x": 2
}

print(dict_sample)

{'x': 2}


In [2]:
dict_sample["y"] = 3

print(dict_sample)


{'x': 2, 'y': 3}


In [7]:
dict_bam_program = {
    "1st year": {
        "1st semester": ["Algebra", "Calculus"],
        "2nd semester": ["Visualization"]
    },
    "2nd year": "blablabla"
}

print(dict_bam_program["1st year"]["1st semester"][1])

Calculus


In [ ]:
list_numbers = [
    [1, 2, 3],
    [4, 5, 6]
]

print(list_numbers[1][2])

In [10]:
empty_set = set()

empty_list = list()
empty_dict = dict()
empty_dict = {}



In [12]:
dict_grades = {
    ("Charles", "Darwin"): 10,
    ("Bad", "Bunny"): 8,
}

print(dict_grades)

{('Charles', 'Darwin'): 10, ('Bad', 'Bunny'): 8}


In [13]:
dict_functions = {
    "yell": print
}

dict_functions["yell"]("Hello")

Hello


In [15]:
dict_functions = {
    print: "Hello"
}

dict_functions

{<function print(*args, sep=' ', end='\n', file=None, flush=False)>: 'Hello'}

In [16]:
dict_grades = {
    ("Charles", "Darwin"): 10,
    ("Bad", "Bunny"): 8,
    (["Jose", "Luis"], "Gomez"): 5,
}

print(dict_grades)

TypeError: unhashable type: 'list'

In [17]:
tuple_with_lists = (["A", "B"], [1, 2])

print(tuple_with_lists)

(['A', 'B'], [1, 2])


In [18]:
tuple_with_lists[0] = ["C", "D"]

print(tuple_with_lists)

TypeError: 'tuple' object does not support item assignment

In [19]:
tuple_with_lists[0][0] = "C"

print(tuple_with_lists)

(['C', 'B'], [1, 2])


# Session 11: Debugging mostly correct Python

AI can produce hundreds or thousands of useful lines quickly. Your responsibility is to recognize the small mistakes that change the result.

This clinic reviews **Modules 1 and 2 only**. It does not use NumPy.


## Learning goals

By the end of the clinic, you should be able to:

- state what correct output should look like before reading every line;
- distinguish an exception from a plausible but incorrect result;
- reduce a problem to a small failing input;
- connect symptoms to types, boundaries, state, scope and data structures;
- make a minimal repair and write a regression test.


## Working method

Work in pairs. Rotate roles after each exercise.

- **Domain expert:** defines the expected behaviour and chooses edge cases.
- **Debugger:** traces the program and proposes the smallest fix.

For every exercise:

1. Predict before executing.
2. Run the supplied tests.
3. Complete the bug report.
4. Change the smallest amount of code.
5. Add at least one new test.

Do not use an AI assistant until your pair has written its prediction and smallest failing input.


## Coverage map

| Concepts | Exercises |
|---|---|
| Numbers, arithmetic and precedence | 1, 8 |
| Strings, slicing and conversion | 2, 4 |
| Conditionals and loops | 3, 6, 7 |
| Functions, returns, annotations and defaults | 1, 4, 7 |
| Lists, tuples, mutability and aliasing | 5, 8 |
| Sets | 6 |
| Dictionaries | 7, 8 |
| Recursion and generators | 9–10, optional |


## Test helper

Run this cell once. The helper reports failures without stopping the rest of the notebook.


In [ ]:
def check_equal(label, function, expected):
    """Run a zero-argument function and print a useful test result."""
    try:
        actual = function()
    except Exception as error:
        print(f"❌ {label}: {type(error).__name__}: {error}")
        return False

    if actual == expected:
        print(f"✅ {label}: {actual!r}")
        return True

    print(f"❌ {label}: expected {expected!r}, obtained {actual!r}")
    return False


def check_raises(label, function, error_type):
    """Pass when a zero-argument function raises the expected exception."""
    try:
        function()
    except error_type:
        print(f"✅ {label}: raised {error_type.__name__}")
        return True
    except Exception as error:
        print(f"❌ {label}: raised {type(error).__name__}, not {error_type.__name__}")
        return False

    print(f"❌ {label}: no exception was raised")
    return False


## Exercise 1: A plausible discount

**Concepts:** numeric variables, arithmetic, operator precedence, functions, default arguments

The function should reduce a price by a proportion. A 10% discount on €100 should return €90. The code runs and produces a plausible number, but the calculation is wrong.


### Predict first

- Expected behaviour:
- Likely smallest failing input:
- Variable or assumption you would inspect first:


In [ ]:
def apply_discount(price: float, discount: float = 0.10) -> float:
    return price * 1 - discount


### Run the evidence


In [ ]:
check_equal("default discount", lambda: apply_discount(100), 90.0)
check_equal("custom discount", lambda: apply_discount(50, 0.15), 42.5)
check_equal("no discount", lambda: apply_discount(25, 0), 25)


### Bug report

- **Observed behaviour:**
- **Expected behaviour:**
- **Smallest failing input:**
- **Violated assumption or invariant:**
- **Root cause:**
- **Minimal fix:**
- **Regression test to add:**


## Exercise 2: The missing digit

**Concepts:** strings, stripping, case conversion, find, slicing, integer conversion

The parser should turn `"The Office S02E12"` into `("The Office", 2, 12)`. It works for single-digit episodes, so the mistake can survive ordinary testing.


### Predict first

- Expected behaviour:
- Likely smallest failing input:
- Variable or assumption you would inspect first:


In [ ]:
def parse_episode(code: str) -> tuple:
    clean = code.strip().upper()
    s_position = clean.rfind("S")
    e_position = clean.find("E", s_position)

    name = clean[:s_position].strip().title()
    season = int(clean[s_position + 1:e_position])
    episode = int(clean[e_position + 1])
    return name, season, episode


### Run the evidence


In [ ]:
check_equal("single-digit episode", lambda: parse_episode("Friends S03E7"), ("Friends", 3, 7))
check_equal("two-digit episode", lambda: parse_episode("The Office S02E12"), ("The Office", 2, 12))
check_equal("surrounding spaces", lambda: parse_episode("  Dark S01E10  "), ("Dark", 1, 10))


### Bug report

- **Observed behaviour:**
- **Expected behaviour:**
- **Smallest failing input:**
- **Violated assumption or invariant:**
- **Root cause:**
- **Minimal fix:**
- **Regression test to add:**


## Exercise 3: A message frozen in time

**Concepts:** loops, conditionals, state changes, f-strings, break-like termination

The function simulates the “take it or double it” game. The returned money is correct, but every message still offers €1.


### Predict first

- Expected behaviour:
- Likely smallest failing input:
- Variable or assumption you would inspect first:


In [ ]:
def simulate_offer(decisions: list[str]) -> tuple:
    value = 1
    messages = []
    message = f"I offer you {value} euros"

    for answer in decisions:
        messages.append(message)
        if answer.lower() == "take it":
            return value, messages
        elif answer.lower() == "double it":
            value *= 2

    return None, messages


### Run the evidence


In [ ]:
decisions = ["double it", "double it", "take it"]
check_equal("value taken", lambda: simulate_offer(decisions)[0], 4)
check_equal(
    "offers shown",
    lambda: simulate_offer(decisions)[1],
    ["I offer you 1 euros", "I offer you 2 euros", "I offer you 4 euros"],
)


### Bug report

- **Observed behaviour:**
- **Expected behaviour:**
- **Smallest failing input:**
- **Violated assumption or invariant:**
- **Root cause:**
- **Minimal fix:**
- **Regression test to add:**


## Exercise 4: A type hint is not a conversion

**Concepts:** function contracts, type hints, strings, float conversion, exceptions

This past-exam-style function accepts a positive number and rounds it without using `round()`. The annotation says `str`, but the caller supplies a float.


### Predict first

- Expected behaviour:
- Likely smallest failing input:
- Variable or assumption you would inspect first:


In [ ]:
def custom_round(num: str) -> int:
    if "." in num:
        integer, decimals = num.split(".")
        if int(decimals[0]) >= 5:
            return int(integer) + 1
        return int(integer)
    return int(num)


### Run the evidence


In [ ]:
check_equal("float below half", lambda: custom_round(3.14159), 3)
check_equal("string above half", lambda: custom_round("4.8"), 5)
check_equal("integer", lambda: custom_round(7), 7)


### Bug report

- **Observed behaviour:**
- **Expected behaviour:**
- **Smallest failing input:**
- **Violated assumption or invariant:**
- **Root cause:**
- **Minimal fix:**
- **Regression test to add:**


## Exercise 5: The tuple changed without changing

**Concepts:** lists, tuples, mutability, aliasing, copying

The function should return an updated roster without modifying the original list. The list is also stored inside a tuple representing a trip.


### Predict first

- Expected behaviour:
- Likely smallest failing input:
- Variable or assumption you would inspect first:


In [ ]:
def add_traveller(roster: list[str], name: str) -> list[str]:
    updated = roster
    updated.append(name)
    return updated


### Run the evidence


In [ ]:
def roster_example():
    original = ["Ana", "Luis"]
    trip = (original, "Madrid")
    updated = add_traveller(original, "Eva")
    return original, updated, trip

check_equal("original list", lambda: roster_example()[0], ["Ana", "Luis"])
check_equal("updated list", lambda: roster_example()[1], ["Ana", "Luis", "Eva"])
check_equal("list inside tuple", lambda: roster_example()[2], (["Ana", "Luis"], "Madrid"))


### Bug report

- **Observed behaviour:**
- **Expected behaviour:**
- **Smallest failing input:**
- **Violated assumption or invariant:**
- **Root cause:**
- **Minimal fix:**
- **Regression test to add:**


## Exercise 6: Attendance without duplicates

**Concepts:** sets, union, intersection, difference, duplicate removal

Produce three sorted lists: students who both attended and submitted, students who attended but did not submit, and students who submitted but did not attend. Duplicate names should disappear.


### Predict first

- Expected behaviour:
- Likely smallest failing input:
- Variable or assumption you would inspect first:


In [ ]:
def attendance_report(attended: list[str], submitted: list[str]) -> tuple:
    attended_set = set(attended)
    submitted_set = set(submitted)

    both = attended_set.union(submitted_set)
    no_submission = submitted_set.difference(attended_set)
    no_attendance = attended_set.difference(submitted_set)

    return sorted(both), sorted(no_submission), sorted(no_attendance)


### Run the evidence


In [ ]:
attended = ["Ana", "Luis", "Ana", "Marta"]
submitted = ["Luis", "Eva", "Eva", "Ana"]

check_equal(
    "attendance report",
    lambda: attendance_report(attended, submitted),
    (["Ana", "Luis"], ["Marta"], ["Eva"]),
)


### Bug report

- **Observed behaviour:**
- **Expected behaviour:**
- **Smallest failing input:**
- **Violated assumption or invariant:**
- **Root cause:**
- **Minimal fix:**
- **Regression test to add:**


## Exercise 7: The dictionary that never starts

**Concepts:** dictionaries, get, loops, return placement, accumulation

Count normalized words. The first version fails on the first unseen word, and a partial repair may still return after processing only one item.


### Predict first

- Expected behaviour:
- Likely smallest failing input:
- Variable or assumption you would inspect first:


In [ ]:
def word_frequencies(words: list[str]) -> dict:
    frequencies = {}

    for word in words:
        clean = word.lower()
        count = frequencies.get(clean)
        frequencies[clean] = count + 1
        return frequencies


### Run the evidence


In [ ]:
check_equal("empty input", lambda: word_frequencies([]), {})
check_equal(
    "repeated words",
    lambda: word_frequencies(["AI", "code", "ai", "Debug"]),
    {"ai": 2, "code": 1, "debug": 1},
)


### Bug report

- **Observed behaviour:**
- **Expected behaviour:**
- **Smallest failing input:**
- **Violated assumption or invariant:**
- **Root cause:**
- **Minimal fix:**
- **Regression test to add:**


## Exercise 8: Capstone: the wrong top student

**Concepts:** dictionaries of lists, loops, averages, integer versus true division, deterministic ties

Return the name with the highest average. Break exact ties alphabetically. The function runs and often gives the correct answer, but truncating averages can create a false tie.


### Predict first

- Expected behaviour:
- Likely smallest failing input:
- Variable or assumption you would inspect first:


In [ ]:
def top_student(grades: dict[str, list[float]]) -> str:
    best_name = ""
    best_average = float("-inf")

    for name, scores in grades.items():
        average = sum(scores) // len(scores)
        if average >= best_average:
            best_name = name
            best_average = average

    return best_name


### Run the evidence


In [ ]:
grades = {
    "Luis": [9.0, 8.9],
    "Ana": [8.9, 8.9],
    "Marta": [8.0, 8.5],
}
check_equal("close averages", lambda: top_student(grades), "Luis")

tied = {"Zoe": [9, 9], "Ana": [9, 9]}
check_equal("alphabetical tie", lambda: top_student(tied), "Ana")


### Bug report

- **Observed behaviour:**
- **Expected behaviour:**
- **Smallest failing input:**
- **Violated assumption or invariant:**
- **Root cause:**
- **Minimal fix:**
- **Regression test to add:**


## Optional: the missing recursive base case

**Concepts:** recursion, base cases, boundary values

The function works for positive integers greater than zero but fails for `factorial(0)`, even though mathematically `0! = 1`.


### Predict first

- Expected behaviour:
- Likely smallest failing input:
- Variable or assumption you would inspect first:


In [ ]:
def factorial(n: int) -> int:
    if n == 1:
        return 1
    return n * factorial(n - 1)


### Run the evidence


In [ ]:
check_equal("factorial of five", lambda: factorial(5), 120)
check_equal("factorial of zero", lambda: factorial(0), 1)
check_raises("negative input", lambda: factorial(-1), ValueError)


### Bug report

- **Observed behaviour:**
- **Expected behaviour:**
- **Smallest failing input:**
- **Violated assumption or invariant:**
- **Root cause:**
- **Minimal fix:**
- **Regression test to add:**


## Optional: the generator stops one step early

**Concepts:** generators, yield, while loops, inclusive boundaries

Yield powers of two up to and including the limit. The code appears correct until the limit itself is a power of two.


### Predict first

- Expected behaviour:
- Likely smallest failing input:
- Variable or assumption you would inspect first:


In [ ]:
def powers_of_two(limit: int):
    value = 1
    while value < limit:
        yield value
        value *= 2


### Run the evidence


In [ ]:
check_equal("limit between powers", lambda: list(powers_of_two(10)), [1, 2, 4, 8])
check_equal("inclusive limit", lambda: list(powers_of_two(8)), [1, 2, 4, 8])


### Bug report

- **Observed behaviour:**
- **Expected behaviour:**
- **Smallest failing input:**
- **Violated assumption or invariant:**
- **Root cause:**
- **Minimal fix:**
- **Regression test to add:**


## Bug exchange

Choose one corrected function and introduce **one subtle bug**.

Your bug must:

- leave the program syntactically valid;
- pass at least one ordinary test;
- fail on an edge case;
- require no more than two changed lines.

Exchange code with another pair. Give them only the requirement and the tests that currently pass. Their job is to discover the failing input, explain the mechanism and add a regression test.


## Exit ticket

Answer individually:

1. Which result today looked plausible but was wrong?
2. What domain knowledge or expected pattern exposed it?
3. What coding intuition helped you locate the cause?
4. Before trusting AI-generated code, what will you test first?


In [ ]:
# Optional: write one final regression test here.
